In [6]:
!pip install xml_to_dict
import requests
from xml_to_dict import XMLtoDict
xd = XMLtoDict()
import pandas as pd
import xml.etree.ElementTree as ET


## 공공데이터 포털 API 방식 시도

In [11]:
# colab에 저장해놓은 개인 API KEY 호출
from google.colab import userdata

api_key = userdata.get('datagov_seibro')

In [36]:
# endcoding
api_key = 'tKyTvg1q3oPeCAXZWW%2FwkdNOUD4Cr9WxJRoNhnTr3F6hje0YvEy7gAheFBx0lcKTZV2R0TV5FBqvnyQ8%2FozvTA%3D%3D'

# decoding
# api_key = 'tKyTvg1q3oPeCAXZWW/wkdNOUD4Cr9WxJRoNhnTr3F6hje0YvEy7gAheFBx0lcKTZV2R0TV5FBqvnyQ8/ozvTA=='

In [33]:
# 고려아연의 예탁결제원 고객번호는 1013
issucoCustno = '593'
secnNm = '고려아연'
rgtStdDt = '20240930'

In [38]:
url

'http://api.seibro.or.kr/openapi/service/StockSvc/getSafeDpDutyDepoStatusN1?stdDt=20250103&listTpcd=12&ServiceKey=tKyTvg1q3oPeCAXZWW%2FwkdNOUD4Cr9WxJRoNhnTr3F6hje0YvEy7gAheFBx0lcKTZV2R0TV5FBqvnyQ8%2FozvTA%3D%3D'

In [37]:
url = f'http://api.seibro.or.kr/openapi/service/StockSvc/getSafeDpDutyDepoStatusN1?stdDt=20250103&listTpcd=12&ServiceKey={api_key}'
raw = requests.get(url)
data_dict = xd.parse(raw.content.decode('utf-8'))
data_list = data_dict['SeibroAPI']['vector']['data']


KeyboardInterrupt: 

In [ ]:
records = []
for item in data_list:
    record = {
        'SHOTN_ISIN': item['result']['SHOTN_ISIN']['@value'],
        'KOR_SECN_NM': item['result']['KOR_SECN_NM']['@value'],
        'ISSUCO_CUSTNO': item['result']['ISSUCO_CUSTNO']['@value']
    }
    records.append(record)
df = pd.DataFrame(records)

## Seibro 홈페이지 직접 크롤링(requests방식)

In [53]:
url = 'https://seibro.or.kr/websquare/engine/proworks/callServletService.jsp'

headers = {
    'Accept': 'application/xml',
    'Accept-Encoding': 'gzip, deflate, br, zstd',
    'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7,ja;q=0.6',
    'Content-Type': 'application/xml; charset="UTF-8"',
    'Origin': 'https://seibro.or.kr',
    'Referer': 'https://seibro.or.kr/websquare/control.jsp?w2xPath=/IPORTAL/user/company/BIP_CNTS01021V.xml&menuNo=19',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36'
}

payload = '''
<reqParam action="termByImptSkedulList" task="ksd.safe.bip.cnts.Company.process.EntrSkedulPTask">
    <MENU_NO value="19"/>
    <ISSUCO_CUSTNO value=""/>
    <DT_TPCD value=""/>
    <FROMDATE value="20250120"/>
    <TODATE value="20250125"/>
    <LIST_TPCD value="12"/>
    <RGT_RACD value=""/>
</reqParam>
'''

response = requests.post(url, headers=headers, data=payload)

if response.status_code == 200:
    print('Successful')
else:
    print(f"Error: {response.status_code}")



Successful


In [54]:
print(response.content.decode('utf-8'))

<?xml version="1.0" encoding="UTF-8" ?>


<WARNING>
	<msg type='java.lang.String' value='서버오류2' />
	<level type='int' value='0' />
	<detail type='java.lang.String' value='' />
	<timestamp type='java.sql.Timestamp' value='1737888931361' />
</WARNING>



In [46]:
# XML 문자열을 파싱 (response.text를 사용한다고 가정)
root = ET.fromstring(response.text)

# 데이터를 저장할 리스트 생성
data_list = []

# 각 데이터 요소에서 정보 추출
for data in root.findall('.//data'):
    result = data.find('result')
    if result is not None:
        item = {}
        for child in result:
            item[child.tag] = child.get('value')
        data_list.append(item)

# 데이터프레임 생성
df = pd.DataFrame(data_list)

# 데이터프레임 출력
df


ParseError: mismatched tag: line 13, column 165 (<string>)

In [44]:
df

,SCH_DTTM,DT_BEGIN_DT,ISSUCO_CUSTNO,SHOTN_CD_ISSUIN_NO,ISSUCO_NM,RGT_RACD,RGT_RANM,RGT_RSN_DTAIL_SORT_CD,RGT_RACD_NM_DETAIL,RGT_STD_DT,DT_TPCD,DT_TPNM,DT_EXPRY_DT,LIST_TPCD,LIST_TPNM,AG_ORG_TPCD,AG_ORG_TPNM,ROST_CLOSE_BEGIN_DT,ROST_CLOSE_EXPRY_DT,STD_DT
0,20250126194117,20240304,1013,01013,고려아연,001,정기총회,N,정기총회,20231231,36,발송통지시한,20240304,11,유가증권시장,01,한국예탁결제원,20240101,20240131,20240304
1,20250126194117,20240319,1013,01013,고려아연,001,정기총회,N,정기총회,20231231,21,총회개최일,20240319,11,유가증권시장,01,한국예탁결제원,20240101,20240131,20240319
2,20250126194117,20240331,1013,01013,고려아연,009,기타,,기타,20240331,01,기준일,20240331,11,유가증권시장,01,한국예탁결제원,,,20240331
3,20250126194117,20240403,1013,01013,고려아연,009,기타,,기타,20240331,34,명세접수시한,20240403,11,유가증권시장,01,한국예탁결제원,,,20240403
4,20250126194117,20240408,1013,01013,고려아연,009,기타,,기타,20240331,35,명세통지시한,20240408,11,유가증권시장,01,한국예탁결제원,,,20240408
5,20250126194117,20240409,1013,01013,고려아연,103,배당/분배,02,배당/분배(현금배당),20231231,10,배당금지급일(1차),20290408,11,유가증권시장,01,한국예탁결제원,20240101,20240131,20290408
6,20250126194117,20240508,1013,01013,고려아연,907,이익소각,99,이익소각,,07,주식발행일,20240508,11,유가증권시장,01,한국예탁결제원,,,20240508
7,20250126194117,20240610,1013,01013,고려아연,907,이익소각,99,이익소각,,08,교부/유통일,20240610,11,유가증권시장,01,한국예탁결제원,,,20240610
8,20250126194117,20240610,1013,01013,고려아연,907,이익소각,99,이익소각,,09,상장일,20240610,11,유가증권시장,01,한국예탁결제원,,,20240610
9,20250126194117,20240627,1013,01013,고려아연,103,배당/분배,02,배당/분배(현금배당),20240630,03,권리락일,20240627,11,유가증권시장,01,한국예탁결제원,,,20240627
